# Tip 5 — Group and Strategy Structure

1. **GROUP as categorical + within-GROUP cross-sectional features** — the four families likely have different return dynamics; ranking within the family is more informative than against the full panel
2. **ALLOCATION target encoding** — smoothed mean of `target` per strategy, computed **inside each CV fold** to avoid leakage; weak signal (~50% base rate) but occasionally informative
3. **MEDIAN_DAILY_TURNOVER interactions** — static liquidity proxy, interacted with vol and volume to modulate flow signals

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set()

import lightgbm as lgbm
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

import tip5_features

### Load data

In [ ]:
X_train = pd.read_csv('../../data/X_train.csv', index_col='ROW_ID')
X_test  = pd.read_csv('../../data/X_test.csv',  index_col='ROW_ID')
y_train = pd.read_csv('../../data/y_train.csv', index_col='ROW_ID')
sample_submission = pd.read_csv('../../data/sample_submission.csv', index_col='ROW_ID')

RET_cols = [f'RET_{i}'           for i in range(1, 21)]
VOL_cols = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]

print("Groups:", X_train['GROUP'].unique())
print("Allocations:", X_train['ALLOCATION'].nunique())

### Feature engineering

#### A. Benchmark features

In [ ]:
def add_benchmark_features(df):
    for i in [3, 5, 10, 15, 20]:
        df[f'AVERAGE_PERF_{i}'] = df[RET_cols[:i]].mean(axis=1)
        df[f'ALLOCATIONS_AVERAGE_PERF_{i}'] = df.groupby('TS')[f'AVERAGE_PERF_{i}'].transform('mean')
    df['STD_PERF_20'] = df[RET_cols].std(axis=1)
    df['ALLOCATIONS_STD_PERF_20'] = df.groupby('TS')['STD_PERF_20'].transform('mean')
    return df

X_train = add_benchmark_features(X_train)
X_test  = add_benchmark_features(X_test)

benchmark_features = (
    RET_cols + VOL_cols + ['MEDIAN_DAILY_TURNOVER']
    + [f'AVERAGE_PERF_{i}'             for i in [3, 5, 10, 15, 20]]
    + [f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3, 5, 10, 15, 20]]
    + ['STD_PERF_20', 'ALLOCATIONS_STD_PERF_20']
)

#### B. Tip 5 — GROUP xs features + TURNOVER interactions (no labels needed)

In [ ]:
X_train, tip5_feats = tip5_features.add_features(X_train, RET_cols, VOL_cols)
X_test,  _          = tip5_features.add_features(X_test,  RET_cols, VOL_cols)

# ALLOC_ENC is computed per fold below — add placeholder to feature list now
tip5_feats_with_enc = tip5_feats + ['ALLOC_ENC']
print(f"Tip 5 features ({len(tip5_feats_with_enc)}):", tip5_feats_with_enc)

### Build final feature list

In [ ]:
features = benchmark_features + tip5_feats_with_enc
print(f"Total features: {len(features)}  "
      f"(benchmark={len(benchmark_features)}, tip5={len(tip5_feats_with_enc)})")

### LightGBM — 8-fold CV with in-fold ALLOCATION target encoding

`ALLOC_ENC` is fit on the training split of each fold and applied to the validation split, so no label leakage occurs.

In [ ]:
lgbm_params = {
    "objective":     "mse",
    "metric":        "mse",
    "num_threads":   50,
    "seed":          42,
    "verbosity":     -1,
    "learning_rate": 1e-2,
    "max_depth":     3,
}
NUM_BOOST_ROUND = 500

train_dates = X_train['TS'].unique()
scores, models = [], []

splits = KFold(n_splits=8, shuffle=True, random_state=0).split(train_dates)

for fold, (tr_idx, val_idx) in enumerate(splits):
    tr_mask  = X_train['TS'].isin(train_dates[tr_idx])
    val_mask = X_train['TS'].isin(train_dates[val_idx])

    y_tr  = y_train.loc[tr_mask,  'target']
    y_val = y_train.loc[val_mask, 'target']

    # Compute ALLOC_ENC on training split only, then apply to both splits
    X_tr_enc  = tip5_features.add_allocation_encoding(
        X_train.loc[tr_mask], y_tr, X_train.loc[tr_mask])
    X_val_enc = tip5_features.add_allocation_encoding(
        X_train.loc[tr_mask], y_tr, X_train.loc[val_mask])

    X_tr  = X_tr_enc[features].fillna(0)
    X_val = X_val_enc[features].fillna(0)

    model = lgbm.train(lgbm_params,
                       lgbm.Dataset(X_tr, label=y_tr.values),
                       num_boost_round=NUM_BOOST_ROUND)

    preds = model.predict(X_val.values, num_threads=lgbm_params['num_threads'])
    acc   = accuracy_score((y_val > 0).astype(int), (preds > 0).astype(int))

    models.append(model)
    scores.append(acc)
    print(f"Fold {fold+1} — Accuracy: {acc*100:.2f}%")

mean = np.mean(scores) * 100
std  = np.std(scores)  * 100
print(f"\nAccuracy: {mean:.2f}% ± {std:.2f}%  [{mean-std:.2f} ; {mean+std:.2f}]")

### Feature importance (top 30 by gain)

In [ ]:
importances = pd.DataFrame(
    [m.feature_importance(importance_type='gain') for m in models],
    columns=features
)
top30 = importances.mean().sort_values(ascending=False).head(30).index

plt.figure(figsize=(10, 9))
sns.barplot(data=importances[top30], orient='h',
            order=importances[top30].mean().sort_values(ascending=False).index)
plt.title("Top-30 features by mean gain (8-fold)")
plt.tight_layout()
plt.show()

### Final model + submission

For the final model, ALLOC_ENC is fit on **all** training data (no fold split needed).

In [ ]:
X_train_enc = tip5_features.add_allocation_encoding(
    X_train, y_train['target'], X_train)
X_test_enc  = tip5_features.add_allocation_encoding(
    X_train, y_train['target'], X_test)

final_model = lgbm.train(
    lgbm_params,
    lgbm.Dataset(X_train_enc[features].fillna(0), label=y_train['target'].values),
    num_boost_round=NUM_BOOST_ROUND
)

test_preds = final_model.predict(X_test_enc[features].fillna(0).values)
submission = pd.DataFrame(
    (test_preds > 0).astype(int),
    index=sample_submission.index,
    columns=['TARGET']
)
submission.to_csv('preds_tip5_group_structure.csv')
print("Saved preds_tip5_group_structure.csv")
submission['TARGET'].value_counts()